# OCR-Engine-Agnostic Post-OCR Correction for Bengali Text Using Small Language Models

**Environment setup and pipeline dry run — Google Colab (CPU-only)**

This notebook prepares the experiment environment and verifies each pipeline stage on a small subset of data before the full evaluation sweep is run. It is intentionally CPU-only, matching the project's core constraint: the correction methods should work on free, commodity hardware, not require a GPU.

**Pipeline overview:**
1. Clone the project code from GitHub.
2. Install dependencies (OCR engines, Bengali language data, model libraries).
3. Load the dataset (British Library historical Bengali corpus) from Google Drive.
4. Run baseline OCR (Tesseract, EasyOCR) on the held-out development pages and verify the CER/WER scoring code against hand-written test cases.
5. Run a single small language model on one page as a sanity check of the zero-shot correction prompt, before committing to the full 5-model × 2-engine × 40-page evaluation.

Project repository: https://github.com/Ariffurrhmn/bengali-postocr-sllm

## 1. Clone the project repository

In [ ]:
!git clone https://github.com/Ariffurrhmn/bengali-postocr-sllm.git
%cd bengali-postocr-sllm

## 2. Install dependencies

Installs the Tesseract OCR engine (system package) and the pinned Python dependencies (`requirements.txt`): `pytesseract`, `easyocr`, `transformers`, `torch`, `jiwer`, and supporting libraries.

In [ ]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr
!pip install -q -r requirements.txt

### Bengali language data for Tesseract

The base Tesseract install does not include Bengali language data. The high-accuracy model (`tessdata_best`) is downloaded into a project-local `.tessdata/` folder, keeping the setup self-contained rather than depending on system-wide configuration.

In [ ]:
!mkdir -p .tessdata
!curl -sL -o .tessdata/ben.traineddata https://github.com/tesseract-ocr/tessdata_best/raw/main/ben.traineddata
!curl -sL -o .tessdata/eng.traineddata https://github.com/tesseract-ocr/tessdata_best/raw/main/eng.traineddata
!curl -sL -o .tessdata/osd.traineddata https://github.com/tesseract-ocr/tessdata_best/raw/main/osd.traineddata

## 3. Authenticate with Hugging Face

Two of the five correction models (Llama 3.2 1B, Gemma 2B) are gated and require an authenticated, license-accepted account to download.

**Before running this cell:** add your Hugging Face access token as a Colab secret (key icon in the left sidebar) named `HF_TOKEN`. This keeps the token out of the notebook's visible code and saved output.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

## 4. Load the dataset

The dataset (British Library historical Bengali corpus, 81 image + PAGE-XML page pairs) is not stored in the code repository. It is uploaded separately to Google Drive as `REID2019.zip` and unzipped here.

**Before running this cell:** upload `REID2019.zip` to a folder named `Dataset` in your Google Drive (`My Drive/Dataset/REID2019.zip`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATASET_ZIP = '/content/drive/My Drive/Dataset/REID2019.zip'
DATASET_DIR = '/content/Competition_dataset_ImagesPAGEXML'

!unzip -q -o "{DATASET_ZIP}" -d /content

import os
n_tif = len([f for f in os.listdir(DATASET_DIR) if f.lower().endswith('.tif')])
n_xml = len([f for f in os.listdir(DATASET_DIR) if f.lower().endswith('.xml')])
print(f'{n_tif} image files, {n_xml} PAGE-XML files found in {DATASET_DIR}')

## 5. Run baseline OCR on the development set

Runs both OCR engines (Tesseract, EasyOCR) over the 10 frozen development pages (`data/split_dev.txt`) and extracts each page's ground truth from its PAGE-XML file. Output is written to `results/ocr_dev.jsonl`.

In [ ]:
os.environ['TESSDATA_PREFIX'] = '/content/bengali-postocr-sllm/.tessdata'
os.environ['TESSERACT_CMD'] = 'tesseract'  # apt-get install puts tesseract on PATH

%cd ocr
!python run_ocr.py --split dev --dataset-dir "{DATASET_DIR}"
%cd ..

## 6. Verify the evaluation code and score the baseline

First runs the hand-written CER/WER test cases (`eval/test_metrics.py`) to confirm the scoring code is correct before trusting it on real data, then scores the raw (uncorrected) OCR output from step 5 against ground truth.

In [ ]:
%cd eval
!python test_metrics.py
print()
!python score_baseline.py --split dev
%cd ..

## 7. Correction dry run

Runs a single model on a single page as a sanity check of the zero-shot correction prompt — catching obvious failure modes (wrong script, refusals, truncation) before committing to the full 5-model × 2-engine × 40-page evaluation sweep.

Starts with `titullm-1b` (Bengali-native, ungated) to avoid depending on gated-model access for this first check. Other valid values: `phi3-mini`, `llama3.2-1b`, `gemma-2b`, `banglat5`.

In [ ]:
%cd correction
!python dry_run.py titullm-1b
%cd ..